# Openings Researcher — Test Notebook

Tests the full pipeline: Lichess ECO lookup → Opening Explorer → web search → LLM synthesis.

In [ ]:
%pip install -q openai tavily-python pydantic requests

In [9]:
import os, sys

PROJECT = r"c:\Users\chris\repos\openings-researcher"
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

print("OPENAI_API_KEY :", "SET" if os.getenv("OPENAI_API_KEY") else "MISSING")
print("TAVILY_API_KEY :", "SET" if os.getenv("TAVILY_API_KEY") else "MISSING")

OPENAI_API_KEY : SET
TAVILY_API_KEY : SET


---
## Module Tests: Lichess Integration

These cells test `lichess.py` directly — no LLM or Tavily calls needed.

In [10]:
from lichess import find_opening, get_variations

SAMPLE = "Ruy Lopez"

entry = find_opening(SAMPLE)
if entry:
    print(f"ECO   : {entry['eco']}")
    print(f"Name  : {entry['name']}")
    print(f"Moves : {entry['pgn']}")
    print(f"FEN   : {entry.get('epd') or entry.get('fen') or '(unavailable)'}")
    print()
    vars_ = get_variations(entry["name"])
    print(f"Variations ({len(vars_)} found):")
    for v in vars_[:8]:
        print(f"  [{v['eco']}]  {v['name']}")
        print(f"         {v['moves']}")
    print()
    print("Raw keys:", list(entry.keys()))
else:
    print(f"No ECO match found for '{SAMPLE}'")

ECO   : C60
Name  : Ruy Lopez
Moves : 1. e4 e5 2. Nf3 Nc6 3. Bb5
FEN   : (unavailable)

Variations (12 found):
  [C60]  Alapin Defense
         1. e4 e5 2. Nf3 Nc6 3. Bb5 Bb4
  [C60]  Alapin Defense, Alapin Gambit
         1. e4 e5 2. Nf3 Nc6 3. Bb5 Bb4 4. c3 Ba5 5. Bxc6 dxc6
  [C60]  Brentano Gambit
         1. e4 e5 2. Nf3 Nc6 3. Bb5 g5
  [C60]  Bulgarian Variation
         1. e4 e5 2. Nf3 Nc6 3. Bb5 a5
  [C60]  Cozio Defense
         1. e4 e5 2. Nf3 Nc6 3. Bb5 Nge7
  [C60]  Cozio Defense, Paulsen Variation
         1. e4 e5 2. Nf3 Nc6 3. Bb5 Nge7 4. Nc3 g6
  [C60]  Cozio Defense, Tartakower Gambit
         1. e4 e5 2. Nf3 Nc6 3. Bb5 g6 4. d4 exd4 5. Nxd4 Bg7 6. Be3 Nge7 7. Nc3 O-O 8. Qd2 d5
  [C60]  Fianchetto Defense
         1. e4 e5 2. Nf3 Nc6 3. Bb5 g6

Raw keys: ['eco', 'name', 'pgn']


In [11]:
from lichess import get_explorer_data

epd = entry.get("epd") or entry.get("fen") if entry else None

if epd:
    data = get_explorer_data(epd, top_games=5)
    if data:
        total = data["white"] + data["draws"] + data["black"]
        print(f"Masters database  ({total:,} games)")
        print(f"  White  {data['white'] / total * 100:.1f}%")
        print(f"  Draws  {data['draws'] / total * 100:.1f}%")
        print(f"  Black  {data['black'] / total * 100:.1f}%")
        print()
        print("Top games:")
        for g in data.get("topGames", []):
            winner = g.get("winner", "")
            result = "1-0" if winner == "white" else ("0-1" if winner == "black" else "½-½")
            print(f"  {g['white']['name']} vs {g['black']['name']}  ({g['year']})  {result}  https://lichess.org/{g['id']}")
    else:
        print("Explorer API returned no data")
elif entry:
    print(f"No FEN/EPD in ECO entry — available keys: {list(entry.keys())}")
else:
    print("Skipped — no ECO entry")

No FEN/EPD in ECO entry — available keys: ['eco', 'name', 'pgn']


In [12]:
from lichess import get_game_pgn
import time

# Debug: walk the full famous-games chain without running the LLM pipeline
if epd:
    raw = get_explorer_data(epd, top_games=5)
    top = raw.get("topGames", []) if raw else []
    print(f"topGames returned: {len(top)}\n")

    for i, g in enumerate(top):
        gid = g.get("id", "")
        winner = g.get("winner", "")
        result = "1-0" if winner == "white" else ("0-1" if winner == "black" else "½-½")
        print(f"[{i}] {g['white']['name']} vs {g['black']['name']}  {g['year']}  {result}")
        print(f"     id={gid}  url=https://lichess.org/{gid}")

        if gid:
            pgn = get_game_pgn(gid)
            if pgn:
                print(f"     PGN: {pgn[:80]}...")
            else:
                print("     PGN: FAILED")
            time.sleep(0.3)
        print()
else:
    print("No EPD — skipping (run ECO cell first)")

No EPD — skipping (run ECO cell first)


---
## Full Pipeline: Single Opening

Runs the complete agent (ECO lookup + Explorer + web search + LLM synthesis) and saves to `test_data/`.

In [13]:
import subprocess

result = subprocess.run(
    [sys.executable, "main.py", "--opening", "Ruy Lopez",
     "--data-dir", "test_data", "--log-level", "DEBUG"],
    cwd=PROJECT, capture_output=True, text=True,
)
print(result.stderr)   # logging goes to stderr
if result.returncode != 0:
    print("STDOUT:", result.stdout)

2026-04-19 15:13:59,579 [INFO] Researching: Ruy Lopez
2026-04-19 15:13:59,581 [DEBUG] Starting new HTTPS connection (1): raw.githubusercontent.com:443
2026-04-19 15:13:59,831 [DEBUG] https://raw.githubusercontent.com:443 "GET /lichess-org/chess-openings/master/a.tsv HTTP/1.1" 200 11115
2026-04-19 15:13:59,840 [DEBUG] Starting new HTTPS connection (1): raw.githubusercontent.com:443
2026-04-19 15:14:00,035 [DEBUG] https://raw.githubusercontent.com:443 "GET /lichess-org/chess-openings/master/b.tsv HTTP/1.1" 200 11103
2026-04-19 15:14:00,045 [DEBUG] Starting new HTTPS connection (1): raw.githubusercontent.com:443
2026-04-19 15:14:00,253 [DEBUG] https://raw.githubusercontent.com:443 "GET /lichess-org/chess-openings/master/c.tsv HTTP/1.1" 200 17848
2026-04-19 15:14:00,271 [DEBUG] Starting new HTTPS connection (1): raw.githubusercontent.com:443
2026-04-19 15:14:00,514 [DEBUG] https://raw.githubusercontent.com:443 "GET /lichess-org/chess-openings/master/d.tsv HTTP/1.1" 200 8430
2026-04-19 15:1

In [14]:
import json
from pathlib import Path

report = json.loads(Path("test_data/ruy_lopez.json").read_text())

# ── core fields ───────────────────────────────────────────────
print(f"Opening : {report['opening']}")
print(f"ECO     : {report['eco_code']}")
print(f"Moves   : {report['moves']}")
print(f"Summary : {report['summary'][:120]}...")
print()

# ── notable players ───────────────────────────────────────────
print("Notable players:", ", ".join(report.get("notable_players", [])))
print()

# ── explorer stats ────────────────────────────────────────────
stats = report.get("explorer_stats")
if stats:
    print(f"Masters DB  ({stats['total_games']:,} games)")
    print(f"  White {stats['white_wins_pct']}%  "
          f"Draws {stats['draws_pct']}%  "
          f"Black {stats['black_wins_pct']}%")
else:
    print("Explorer stats: unavailable")
print()

# ── variations ────────────────────────────────────────────────
variations = report.get("variations", [])
print(f"Variations ({len(variations)}):")
for v in variations[:6]:
    print(f"  [{v['eco']}]  {v['name']}")
    print(f"         {v['moves']}")
print()

# ── famous games ─────────────────────────────────────────────
games = report.get("famous_games", [])
print(f"Famous games ({len(games)}):")
for g in games:
    pgn_snippet = g['pgn'][:60] + "..." if g.get("pgn") else "(no PGN)"
    print(f"  {g['white']} vs {g['black']}  ({g['year']})  {g['result']}")
    print(f"    {g['lichess_url']}")
    print(f"    {pgn_snippet}")
print()

# ── article links ─────────────────────────────────────────────
links = report.get("article_links", [])
print(f"Article links ({len(links)}):")
for a in links[:8]:
    print(f"  {a['title']}")
    print(f"    {a['url']}")

Opening : Ruy Lopez
ECO     : C60
Moves   : 1. e4 e5 2. Nf3 Nc6 3. Bb5
Summary : The Ruy Lopez, also known as the Spanish Opening, is one of the oldest and most respected chess openings, beginning with...

Notable players: Wilhelm Steinitz, Bobby Fischer, Vladimir Kramnik, Garry Kasparov, Magnus Carlsen, José Raúl Capablanca, Anatoly Karpov, Viswanathan Anand

Explorer stats: unavailable

Variations (12):
  [C60]  Alapin Defense
         1. e4 e5 2. Nf3 Nc6 3. Bb5 Bb4
  [C60]  Alapin Defense, Alapin Gambit
         1. e4 e5 2. Nf3 Nc6 3. Bb5 Bb4 4. c3 Ba5 5. Bxc6 dxc6
  [C60]  Brentano Gambit
         1. e4 e5 2. Nf3 Nc6 3. Bb5 g5
  [C60]  Bulgarian Variation
         1. e4 e5 2. Nf3 Nc6 3. Bb5 a5
  [C60]  Cozio Defense
         1. e4 e5 2. Nf3 Nc6 3. Bb5 Nge7
  [C60]  Cozio Defense, Paulsen Variation
         1. e4 e5 2. Nf3 Nc6 3. Bb5 Nge7 4. Nc3 g6

Famous games (0):

Article links (19):
  Ruy Lopez: How to Play It, Attack It & Beat It - Royal Chess Mall
    https://royalchessmall.c

In [15]:
# Raw JSON — full output
print(json.dumps(report, indent=2))

{
  "opening": "Ruy Lopez",
  "eco_code": "C60",
  "moves": "1. e4 e5 2. Nf3 Nc6 3. Bb5",
  "summary": "The Ruy Lopez, also known as the Spanish Opening, is one of the oldest and most respected chess openings, beginning with 1.e4 e5 2.Nf3 Nc6 3.Bb5. It aims for long-term positional pressure by pinning Black's knight and controlling the center. The opening leads to rich strategic and tactical possibilities, with White focusing on central control, rapid development, and kingside attacks, while Black counters with queenside expansion and central counterplay. Variations such as the Berlin Defense have become prominent at the highest levels. The Ruy Lopez remains a staple in the repertoires of many world champions and grandmasters.",
  "ideas": [
    "White pins Black's knight on c6 to indirectly pressure the e5 pawn.",
    "White aims to control the center with pawns on d4 and e4, supported by c3.",
    "White develops pieces rapidly, often preparing kingside attacks with maneuvers like Nb

---
## Batch Test

Research two openings at once and inspect the manifest.

In [ ]:
import json

test_openings = ["Sicilian Defense", "King's Indian Defense"]
Path("test_openings.json").write_text(json.dumps(test_openings))

result = subprocess.run(
    [sys.executable, "main.py",
     "--openings-file", "test_openings.json",
     "--data-dir", "test_data",
     "--request-delay", "0.5"],
    cwd=PROJECT, capture_output=True, text=True,
)
print(result.stderr)
if result.returncode != 0:
    print("STDOUT:", result.stdout)

In [ ]:
manifest = json.loads(Path("test_data/_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
print()
print("Files in test_data/:")
for p in sorted(Path("test_data").iterdir()):
    print(f"  {p.name}  ({p.stat().st_size:,} bytes)")

In [ ]:
import shutil

shutil.rmtree("test_data", ignore_errors=True)
Path("test_openings.json").unlink(missing_ok=True)
print("Cleaned up.")